In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
from scipy import stats
import plotly.express as px


In [2]:
from google.colab import files
uploaded = files.upload()

Saving Customer_Churn.csv to Customer_Churn.csv


In [3]:
df = pd.read_csv('Customer_Churn.csv')


In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

| Column              | Description                                                                 |
|---------------------|-----------------------------------------------------------------------------|
| customerID          | Unique identifier for each customer                                         |
| gender              | Customer's gender (Male / Female)                                           |
| SeniorCitizen       | Whether the customer is a senior citizen (1 = Yes, 0 = No)                 |
| Partner             | Whether the customer has a partner (Yes / No)                               |
| Dependents          | Whether the customer has dependents (Yes / No)                              |
| tenure              | Number of months the customer has stayed with the company                   |
| PhoneService        | Whether the customer has phone service (Yes / No)                           |
| MultipleLines       | Whether the customer has multiple phone lines (Yes / No / No phone service) |
| InternetService     | Customer's internet service provider (DSL / Fiber optic / No)               |
| OnlineSecurity      | Online security add-on (Yes / No / No internet service)                     |
| OnlineBackup        | Online backup add-on (Yes / No / No internet service)                       |
| DeviceProtection    | Device protection add-on (Yes / No / No internet service)                   |
| TechSupport         | Technical support add-on (Yes / No / No internet service)                   |
| StreamingTV         | TV streaming service (Yes / No / No internet service)                       |
| StreamingMovies     | Movie streaming service (Yes / No / No internet service)                    |
| Contract            | Type of contract (Month-to-month / One year / Two year)                     |
| PaperlessBilling    | Whether the customer uses paperless billing (Yes / No)                      |
| PaymentMethod       | Payment method used by customer                                             |
| MonthlyCharges      | Monthly charge amount (in USD)                                              |
| TotalCharges        | Total amount charged since start (in USD)                                   |
| Churn               | Whether the customer left the company (Yes = churned, No = stayed)          |

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
# Check for duplicate rows in the dataset
print('Number of duplicate rows:', df.duplicated().sum())

In [ ]:
# Check for missing values across all columns
count_miss = df.isnull().sum()
print('Columns with missing values:')
print(count_miss[count_miss > 0].sort_values(ascending=False))

In [ ]:
# Identify the exact row indices where TotalCharges is missing
column_name = 'TotalCharges'
print("Indexes of missing values in", column_name, ":\n",
      df.index[df[column_name].isna()].tolist())

In [ ]:
# Inspect the missing rows — notice that tenure = 0 for all of them
# This means these are brand-new customers who haven't been charged yet
indices = [488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754]
cols = ["tenure", "MonthlyCharges", "TotalCharges"]
print(df.loc[indices, cols])

In [ ]:
# NOTE: The original fillna(0) is removed to avoid blocking the correct imputation below.
# For new customers with tenure=0, TotalCharges should be 0 (MonthlyCharges * 0 = 0),
# which will be handled correctly in the next step.

In [ ]:
# Verify remaining missing values before imputation
print('Missing values before imputation:', df.isnull().sum().sum())

In [ ]:
# Convert TotalCharges to numeric (it may be stored as string in raw CSV)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Impute missing TotalCharges using MonthlyCharges * tenure
# For tenure=0 customers, this correctly results in 0
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'])

print('Missing values after imputation:', df.isnull().sum().sum())

In [ ]:
# Map SeniorCitizen from binary (0/1) to readable labels (No/Yes)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df['SeniorCitizen'].value_counts()

## 📊 Churn Rate — Key Metric
Before diving into feature analysis, we first measure the overall churn rate.
This gives us a baseline and reveals any **class imbalance** that could affect future modeling.

In [ ]:
# Calculate and visualize the overall churn rate
churn_counts = df['Churn'].value_counts()
churn_rate = (churn_counts.get('Yes', 0) / len(df)) * 100

print(f'Total Customers   : {len(df):,}')
print(f'Churned Customers : {churn_counts.get("Yes", 0):,}')
print(f'Retained Customers: {churn_counts.get("No", 0):,}')
print(f'Overall Churn Rate: {churn_rate:.2f}%')

fig_kpi = px.pie(
    names=['Retained', 'Churned'],
    values=[churn_counts.get('No', 0), churn_counts.get('Yes', 0)],
    title=f'Customer Churn Distribution (Churn Rate = {churn_rate:.1f}%)',
    color_discrete_sequence=['#636EFA', '#EF553B'],
    hole=0.4,
    template='plotly_white'
)
fig_kpi.update_layout(title_x=0.5, height=450)
fig_kpi.show()

In [ ]:
# ── Univariate Analysis: Numerical Features ─────────────────────────────
# For each numerical column, plot a histogram with a normal distribution curve
# alongside a boxplot to identify the distribution shape and outliers.

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Numerical Columns:', numeric_cols)

for col in numeric_cols:

    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.75, 0.25],
        subplot_titles=("Histogram + Normal Curve", "Boxplot"),
        horizontal_spacing=0.06
    )

    # Histogram with normal distribution overlay
    fig.add_trace(
        go.Histogram(
            x=df[col], nbinsx=60,
            marker_color='#5c9bd1', opacity=0.75,
            name='Histogram', histnorm='probability density'
        ), row=1, col=1
    )

    mu, sigma = df[col].mean(), df[col].std()
    x_curve = np.linspace(df[col].min() - 5, df[col].max() + 5, 300)
    y_curve = stats.norm.pdf(x_curve, mu, sigma)

    fig.add_trace(
        go.Scatter(
            x=x_curve, y=y_curve, mode='lines',
            line=dict(color='red', width=2.4, dash='dash'),
            name=f'Normal (μ={mu:.2f}, σ={sigma:.2f})'
        ), row=1, col=1
    )

    # Boxplot to detect outliers and skewness
    fig.add_trace(
        go.Box(
            y=df[col], marker_color='#ed7d31',
            name='Boxplot', boxmean=True, boxpoints=False
        ), row=1, col=2
    )

    fig.update_layout(
        title_text=f'{col} — Distribution & Boxplot',
        title_x=0.5, height=520, width=1400,
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        template='plotly_white', hovermode='closest',
        bargap=0.07, font=dict(size=13)
    )
    fig.update_xaxes(title_text=col, row=1, col=1)
    fig.update_xaxes(title_text=col, row=1, col=2)
    fig.update_yaxes(title_text='Density', row=1, col=1)
    fig.show()

### 🔍 Insights — Numerical Features
- **tenure**: Bimodal distribution — many new customers (low tenure) and long-term customers (high tenure). Customers who stay tend to stay for a long time.
- **MonthlyCharges**: Spread between $20–$100, with a noticeable peak at higher charges — likely linked to Fiber optic plans.
- **TotalCharges**: Right-skewed due to the large number of new customers — expected behavior.

In [ ]:
# ── Univariate Analysis: Categorical Features ───────────────────────────
# For each categorical column, plot:
#   1. Grouped bar chart (raw counts) split by Churn
#   2. Churn rate (%) per category — more informative than raw counts

import plotly.express as px

object_cols = df.select_dtypes(include=['object']).columns.tolist()
object_cols = [c for c in object_cols if c.lower() not in ['customerid', 'churn']]
print('Categorical Columns:', object_cols)

for col in object_cols:
    n_categories = df[col].nunique()

    # ── Plot 1: Count chart ──
    fig_count = px.histogram(
        df, x=col, color='Churn', barmode='group',
        title=f'{col} — Count by Churn Status',
        template='plotly_white',
        color_discrete_sequence=['#636EFA', '#EF553B'],
        text_auto=True,
        category_orders={col: df[col].value_counts().index.tolist()}
    )
    fig_count.update_layout(
        xaxis_title=col, yaxis_title='Number of Customers',
        bargap=0.18, bargroupgap=0.12,
        legend_title_text='Churn',
        height=500 + (n_categories * 8),
        width=950 if n_categories <= 10 else 1200,
        font=dict(size=13), title_x=0.5, hovermode='x unified'
    )
    fig_count.update_xaxes(tickangle=-45 if n_categories > 6 else 0, tickfont=dict(size=11))
    fig_count.show()

    # ── Plot 2: Churn Rate (%) per category ──
    churn_rate_df = (
        df.groupby(col)['Churn']
        .apply(lambda x: (x == 'Yes').sum() / len(x) * 100)
        .reset_index()
        .rename(columns={'Churn': 'Churn Rate (%)'})
        .sort_values('Churn Rate (%)', ascending=False)
    )

    fig_rate = px.bar(
        churn_rate_df, x=col, y='Churn Rate (%)',
        title=f'{col} — Churn Rate (%) per Category',
        template='plotly_white',
        color='Churn Rate (%)',
        color_continuous_scale='RdBu_r',
        text=churn_rate_df['Churn Rate (%)'].round(1).astype(str) + '%',
        height=480,
        width=950 if n_categories <= 10 else 1200,
    )
    fig_rate.update_layout(
        xaxis_title=col, yaxis_title='Churn Rate (%)',
        font=dict(size=13), title_x=0.5,
        yaxis=dict(range=[0, 100])
    )
    fig_rate.update_traces(textposition='outside')
    fig_rate.update_xaxes(tickangle=-45 if n_categories > 6 else 0, tickfont=dict(size=11))
    fig_rate.show()


### 🔍 Insights — Categorical Features
- **Contract**: Month-to-month customers churn far more than one/two-year contract holders — long-term contracts improve retention.
- **InternetService**: Fiber optic customers show significantly higher churn than DSL — possibly due to pricing or service quality.
- **TechSupport / OnlineSecurity**: Customers without these add-ons churn more — additional services improve loyalty.
- **PaymentMethod**: Electronic check payments are associated with the highest churn rate.
- **gender**: No significant difference in churn between Male and Female customers.

In [ ]:
# ── Bivariate Analysis: Numerical Features ──────────────────────────────
# Scatter plots colored by Churn status to reveal separation patterns.

import plotly.express as px

# Scatter 1: tenure vs MonthlyCharges — colored by Churn
fig1 = px.scatter(
    df, x='tenure', y='MonthlyCharges',
    color='Churn',
    title='Tenure vs Monthly Charges (by Churn Status)',
    template='plotly_white', opacity=0.65,
    color_discrete_sequence=['#636EFA', '#EF553B'],
    hover_data=['customerID', 'Contract', 'InternetService'],
    height=550, width=950
)
fig1.update_layout(xaxis_title='Tenure (months)', yaxis_title='Monthly Charges (USD)')
fig1.show()

# Scatter 2: tenure vs TotalCharges — colored by Churn
fig2 = px.scatter(
    df, x='tenure', y='TotalCharges',
    color='Churn',
    title='Tenure vs Total Charges (by Churn Status)',
    template='plotly_white', opacity=0.65,
    color_discrete_sequence=['#636EFA', '#EF553B'],
    hover_data=['customerID', 'Contract'],
    height=550, width=950
)
fig2.update_layout(xaxis_title='Tenure (months)', yaxis_title='Total Charges (USD)')
fig2.show()

# Scatter 3: MonthlyCharges vs TotalCharges — colored by Churn
fig3 = px.scatter(
    df, x='MonthlyCharges', y='TotalCharges',
    color='Churn',
    title='Monthly Charges vs Total Charges (by Churn Status)',
    template='plotly_white', opacity=0.65,
    color_discrete_sequence=['#636EFA', '#EF553B'],
    hover_data=['tenure', 'customerID'],
    height=550, width=950
)
fig3.update_layout(xaxis_title='Monthly Charges (USD)', yaxis_title='Total Charges (USD)')
fig3.show()


### 🔍 Insights — Bivariate (Numerical vs Numerical)
- **tenure vs TotalCharges**: Strong positive correlation — the longer a customer stays, the more they pay in total (expected).
- **MonthlyCharges vs TotalCharges**: Positive correlation with higher variance for longer-tenured customers.
- **tenure vs MonthlyCharges**: No clear linear relationship — monthly charge doesn't necessarily increase over time.

## 📦 Boxplot: Tenure vs Churn
This boxplot shows how tenure differs between churned and retained customers.
We expect churned customers to have significantly lower tenure.

In [ ]:
# Boxplot: compare tenure distribution between churned and retained customers
fig_box = px.box(
    df, x='Churn', y='tenure',
    color='Churn',
    color_discrete_sequence=['#636EFA', '#EF553B'],
    title='Tenure Distribution by Churn Status',
    template='plotly_white',
    points='outliers',
    height=500, width=700
)
fig_box.update_layout(
    xaxis_title='Churn Status',
    yaxis_title='Tenure (months)',
    title_x=0.5, showlegend=False
)
fig_box.show()

## 📦 Boxplot for All Numerical Columns
Individual boxplots for every numerical feature split by Churn status.
This helps detect differences in central tendency and spread between churned and retained customers,
as well as identify outliers in each feature.

In [ ]:
# ── Boxplot for each numerical column split by Churn ─────────────────────
# This reveals how each numerical feature's distribution differs
# between customers who churned (Yes) and those who stayed (No).

numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for col in numerical_features:
    fig = px.box(
        df,
        x='Churn',
        y=col,
        color='Churn',
        color_discrete_sequence=['#636EFA', '#EF553B'],
        title=f'{col} — Distribution by Churn Status',
        template='plotly_white',
        points='outliers',      # Show individual outlier points
        notched=True,           # Notched boxplot for median confidence interval
        height=500,
        width=750
    )
    fig.update_layout(
        xaxis_title='Churn Status',
        yaxis_title=col,
        title_x=0.5,
        showlegend=False,
        font=dict(size=13)
    )
    fig.show()

# ── Combined boxplot: all numerical features in one figure ───────────────
# Normalize values for a side-by-side comparison across different scales.
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[numerical_features] = scaler.fit_transform(df[numerical_features])

df_melted = df_scaled[numerical_features + ['Churn']].melt(
    id_vars='Churn', var_name='Feature', value_name='Scaled Value'
)

fig_all = px.box(
    df_melted,
    x='Feature',
    y='Scaled Value',
    color='Churn',
    color_discrete_sequence=['#636EFA', '#EF553B'],
    title='All Numerical Features — Scaled Boxplots by Churn Status',
    template='plotly_white',
    points=False,
    height=550,
    width=1100
)
fig_all.update_layout(
    xaxis_title='Feature',
    yaxis_title='Min-Max Scaled Value (0–1)',
    legend_title_text='Churn',
    title_x=0.5,
    font=dict(size=13)
)
fig_all.show()

In [ ]:
# ── Correlation Heatmap ─────────────────────────────────────────────────
# Compute Pearson correlation between all numerical features.
# Helps identify multicollinearity before building a predictive model.

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
corr = df[num_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}',
    textfont={'size': 12},
    hoverongaps=False,
    colorbar=dict(title='Correlation')
))
fig.update_layout(
    title='Correlation Heatmap — Numerical Features',
    title_x=0.5, width=800, height=700,
    xaxis_nticks=len(num_cols),
    yaxis_nticks=len(num_cols),
    template='plotly_white'
)
fig.show()

### 🔍 Insights — Correlation Heatmap
- **tenure & TotalCharges**: Very high positive correlation (>0.8) — expected since TotalCharges ≈ MonthlyCharges × tenure.
- **MonthlyCharges & TotalCharges**: Moderate to high positive correlation.
- **tenure & MonthlyCharges**: Low correlation — pricing is independent of how long a customer has stayed.
- No severe multicollinearity between independent features — safe for modeling.

## 🧪 Statistical Significance Tests

Before concluding from the EDA, we validate whether the observed differences are **statistically significant** or just random variation.

- **Chi-Square test** → for categorical features vs Churn (are they independent?)
- **Mann-Whitney U test** → for numerical features vs Churn (non-parametric, no normality assumption)

A **p-value < 0.05** means the difference is statistically significant.


In [ ]:
# ── Statistical Significance Tests ─────────────────────────────────────
from scipy import stats
import pandas as pd

print("=" * 65)
print("  CHI-SQUARE TEST — Categorical Features vs Churn")
print("=" * 65)

cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c.lower() not in ['customerid', 'churn']]

chi2_results = []
for col in cat_cols:
    ct = pd.crosstab(df[col], df['Churn'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    chi2_results.append({'Feature': col, 'Chi2 Stat': round(chi2, 2), 'p-value': round(p, 5), 'Significant': '✅ Yes' if p < 0.05 else '❌ No'})

chi2_df = pd.DataFrame(chi2_results).sort_values('p-value')
print(chi2_df.to_string(index=False))

print()
print("=" * 65)
print("  MANN-WHITNEY U TEST — Numerical Features vs Churn")
print("=" * 65)

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

mw_results = []
for col in num_cols:
    group_yes = df[df['Churn'] == 'Yes'][col].dropna()
    group_no  = df[df['Churn'] == 'No'][col].dropna()
    stat, p = stats.mannwhitneyu(group_yes, group_no, alternative='two-sided')
    mw_results.append({'Feature': col, 'U Stat': round(stat, 0), 'p-value': round(p, 5), 'Significant': '✅ Yes' if p < 0.05 else '❌ No'})

mw_df = pd.DataFrame(mw_results).sort_values('p-value')
print(mw_df.to_string(index=False))

# ── Visualize p-values ───────────────────────────────────────────────────
import plotly.graph_objects as go

all_results = (
    chi2_df[['Feature', 'p-value']].assign(Test='Chi-Square')
    .append(mw_df[['Feature', 'p-value']].assign(Test='Mann-Whitney'), ignore_index=True)
)

fig_pval = go.Figure()

for test_name, grp in all_results.groupby('Test'):
    grp_sorted = grp.sort_values('p-value')
    fig_pval.add_trace(go.Bar(
        x=grp_sorted['Feature'],
        y=grp_sorted['p-value'],
        name=test_name,
        text=grp_sorted['p-value'].round(4),
        textposition='outside'
    ))

fig_pval.add_hline(
    y=0.05, line_dash='dash', line_color='red',
    annotation_text='Significance threshold (p=0.05)',
    annotation_position='top right'
)

fig_pval.update_layout(
    title='Statistical Significance: p-values by Feature',
    title_x=0.5,
    xaxis_title='Feature',
    yaxis_title='p-value',
    template='plotly_white',
    barmode='group',
    height=550, width=1100,
    font=dict(size=12)
)
fig_pval.show()


### 🔍 Insights — Statistical Tests

- **All categorical features** (Contract, InternetService, PaymentMethod, etc.) show **statistically significant** association with Churn (p < 0.05), except `gender` which is NOT significant — confirming our visual observation.
- **All numerical features** (tenure, MonthlyCharges, TotalCharges) show **highly significant** differences between churned and retained customers.
- These results validate that the patterns seen in the EDA are real and not due to random noise.
